# Feature engineering

The original notebook noticed that adding batting average (BA) to a
regression alongside OBP and SLG flips its coefficient negative, a
classic multicollinearity symptom, and fixed it by dropping BA based on
intuition ("least significant, drop it"). This puts an actual variance
inflation factor on that decision, and builds the feature sets used for
modeling in the next notebook.

In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

baseball = pd.read_csv('baseball.csv')
baseball['RD'] = baseball.RS - baseball.RA

In [2]:
X_full = sm.add_constant(baseball[['OBP', 'SLG', 'BA']])
vif = pd.Series(
    [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])],
    index = X_full.columns
)
vif

const    590.665922
OBP        4.212412
SLG        3.078599
BA         4.204787
dtype: float64

BA's VIF (4.2) is well below the commonly-cited VIF > 10 red flag, so this isn't the kind of severe multicollinearity seen elsewhere in this portfolio (Wine-Linear-Regression's Age/FrancePop pair hit VIF 98). It's still elevated enough to be worth removing, a moderate, real overlap between batting average and the two stats that already incorporate it, but this is a case where the original's "drop it, it's redundant" instinct was directionally right without the underlying multicollinearity actually being as severe as the flipped sign made it look.

In [3]:
X_reduced = sm.add_constant(baseball[['OBP', 'SLG']])
vif_reduced = pd.Series(
    [variance_inflation_factor(X_reduced.values, i) for i in range(X_reduced.shape[1])],
    index = X_reduced.columns
)
vif_reduced

const    547.482728
OBP        2.670503
SLG        2.670503
dtype: float64

In [4]:
import os
os.makedirs('data', exist_ok = True)
baseball.to_pickle('data/baseball_clean.pkl')

import json
with open('outputs/feature_engineering_summary.json', 'w') as f:
    json.dump({
        'vif_ba_with_obp_slg': float(vif['BA']),
        'vif_obp_reduced': float(vif_reduced['OBP']),
        'vif_slg_reduced': float(vif_reduced['SLG']),
    }, f, indent = 2)